In [6]:
%matplotlib inline
import scanpy as sc
import scrublet as scr
import scipy.io
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
from sklearn.cluster import KMeans
import glob

In [2]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rc('font', size=14)
plt.rcParams['pdf.fonttype'] = 42

In [4]:
def run_custom_scrublet(count_matrix, expected_doublet_rate=0.15, n_iterations=10):
    """
    根据文献描述的 10 次迭代 KMeans 聚类法寻找最佳 Scrublet threshold。
    
    参数:
    count_matrix: 细胞 x 基因的原始 UMI count 矩阵 (未标准化, scipy.sparse 或 numpy.ndarray)
    expected_doublet_rate: 预期的 doublet 比例，默认为 0.15
    n_iterations: 迭代次数，默认为 10
    
    返回:
    doublet_scores: 每个细胞的 doublet score
    predicted_doublets: 每个细胞是否为 doublet (布尔值)
    final_cutoff: 最终计算出的平均阈值
    """
    boundaries = []
    final_scrub_obj = None
    
    for i in range(n_iterations):
        # 1. 设置 random_state
        # "In the final iteration, the random_state of the scrublet object is set to 0."
        # 否则设置为 0 到 1,000,000 之间的随机整数
        if i == n_iterations - 1:
            current_random_state = 0
        else:
            current_random_state = random.randint(0, 1000000)
            
        print(f"Iteration {i+1}/{n_iterations} - random_state: {current_random_state}")
        
        # 2. 初始化 Scrublet 对象
        scrub = scr.Scrublet(count_matrix, 
                             expected_doublet_rate=expected_doublet_rate, 
                             random_state=current_random_state)
        
        # 保存最后一次迭代的 Scrublet 对象，用于最后的 call_doublets
        if i == n_iterations - 1:
            final_scrub_obj = scrub
            
        # 3. 计算 doublet scores
        # parameters: min_counts=2, log_transform=True
        doublet_scores, _ = scrub.scrub_doublets(min_counts=2, log_transform=True)
        
        # 4. 使用 KMeans 对 doublet scores 进行聚类 (n_clusters=2)
        # KMeans 需要 2D 数组输入，所以将 1D 的 scores 转换一下形状
        scores_2d = doublet_scores.reshape(-1, 1)
        
        kmeans = KMeans(n_clusters=2, 
                        init='k-means++', 
                        n_init=10, 
                        max_iter=10000,
                        random_state=current_random_state) # 保持随机种子一致性
        labels = kmeans.fit_predict(scores_2d)
        
        # 5. 寻找 singlet 和 doublet 聚类之间的边界 (Boundary)
        # 首先区分哪个是 singlet cluster (得分低的)，哪个是 doublet cluster (得分高的)
        cluster_centers = kmeans.cluster_centers_.flatten()
        singlet_cluster_idx = np.argmin(cluster_centers)
        doublet_cluster_idx = np.argmax(cluster_centers)
        
        # 提取两类的 scores
        singlet_scores = doublet_scores[labels == singlet_cluster_idx]
        doublet_cluster_scores = doublet_scores[labels == doublet_cluster_idx]
        
        # 边界定义为：Singlet 的最高分与 Doublet 的最低分的中点
        max_singlet = np.max(singlet_scores)
        min_doublet = np.min(doublet_cluster_scores)
        boundary = (max_singlet + min_doublet) / 2.0
        
        boundaries.append(boundary)
        print(f"  -> Boundary for iteration {i+1}: {boundary:.4f}")

    # 6. 计算所有迭代的平均边界，作为最终的 cut-off
    final_cutoff = np.mean(boundaries)
    print(f"\nFinal averaged doublet cut-off: {final_cutoff:.4f}")
    
    # 7. 使用最终的 cutoff 预测 doublet 状态
    # "used by the Scrublet call_doublets function to predict a cell’s doublet status"
    predicted_doublets = final_scrub_obj.call_doublets(threshold=final_cutoff)
    
    return final_scrub_obj, predicted_doublets, final_cutoff

In [7]:
# 2. 设置输入和输出目录
input_dir = '/mnt/netshare1/wangzhen/cancer_data/HTAN/cancer_multiome/synapse/'
output_dir = '/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/1_processed_data/'

# 确保输出目录存在，如果不存在则自动创建
os.makedirs(output_dir, exist_ok=True)
# 3. 自动获取目录下所有的 matrix 文件，提取前缀
matrix_files = glob.glob(os.path.join(input_dir, '*matrix.mtx.gz'))
# 使用列表推导式提取前缀 (例如 'CE336E1-S1-')
prefixes = [os.path.basename(f).replace('matrix.mtx.gz', '') for f in matrix_files]

print(f"总共检测到 {len(prefixes)} 个样本，准备开始批量计算...\n")

总共检测到 136 个样本，准备开始批量计算...



In [8]:
# 4. 开启批量循环
for prefix in prefixes:
    print(f"\n=======================================================")
    print(f"正在处理样本: {prefix}")
    print(f"=======================================================")
    
    try:
        # 读入当前前缀的数据
        adata = sc.read_10x_mtx(
            input_dir, 
            var_names='gene_ids', 
            prefix=prefix,      
            gex_only=False
        )
        
        adata_multi = adata.copy()
        print(f"  原始 Barcode 总数: {adata_multi.n_obs}")

        # 拆分 RNA 和 ATAC 模态
        adata_rna = adata_multi[:, adata_multi.var['feature_types'] == 'Gene Expression'].copy()
        adata_atac = adata_multi[:, adata_multi.var['feature_types'] == 'Peaks'].copy()

        # 模拟初步过滤 (阈值 >= 100)
        rna_counts = np.array(adata_rna.X.sum(axis=1)).flatten()
        atac_counts = np.array(adata_atac.X.sum(axis=1)).flatten()
        valid_barcodes_mask = (rna_counts >= 100) & (atac_counts >= 100)

        valid_barcodes = adata_multi.obs_names[valid_barcodes_mask]
        print(f"  初滤后(>=100 counts) 剩余真实细胞数: {len(valid_barcodes)}")

        # 如果细胞数过少（比如小于 10 个），跳过该样本以防 Scrublet 报错
        if len(valid_barcodes) < 10:
            print(f"  ⚠️ 样本 {prefix} 过滤后细胞数极低，跳过。")
            continue

        adata_rna_filtered = adata_rna[valid_barcodes].copy()
        adata_atac_filtered = adata_atac[valid_barcodes].copy()

        # 运行 Scrublet
        print("  --> 计算 RNA 模态的 Doublets...")
        # 修复：直接接收 scores_rna
        scrub_rna, is_doublet_rna, threshold_rna = run_custom_scrublet(adata_rna_filtered.X, n_iterations=10)

        print("  --> 计算 ATAC 模态的 Doublets...")
        # 修复：直接接收 scores_atac
        scrub_atac, is_doublet_atac, threshold_atac = run_custom_scrublet(adata_atac_filtered.X, n_iterations=10)

        # 取交集
        final_is_doublet = is_doublet_rna & is_doublet_atac

        # 组装结果并导出 CSV，保存为动态生成的文件名
        results_df = pd.DataFrame({
            'doublet_RNA': is_doublet_rna,
            'doublet_ATAC': is_doublet_atac,
            'predicted_doublet_final': final_is_doublet
        }, index=valid_barcodes) 

        # 输出文件路径拼接，例如 CE336E1-S1-_Scrublet_Results.csv
        output_csv = os.path.join(output_dir, f"{prefix}_Scrublet_Results.csv")
        results_df.to_csv(output_csv)

        print(f"  ✅ {prefix} 处理完成！")
        print(f"     RNA 判定的 Doublet 数量: {sum(is_doublet_rna)}")
        print(f"     ATAC 判定的 Doublet 数量: {sum(is_doublet_atac)}")
        print(f"     两边共同判定的最终 Doublet: {sum(final_is_doublet)}")
        print(f"     结果已保存至: {output_csv}\n")
        
    except Exception as e:
        print(f"  ❌ 样本 {prefix} 在处理时发生错误: {e}")


正在处理样本: CE357E1-S1-
  原始 Barcode 总数: 708235
  初滤后(>=100 counts) 剩余真实细胞数: 14853
  --> 计算 RNA 模态的 Doublets...
Iteration 1/10 - random_state: 793074
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.19
Detected doublet rate = 23.9%
Estimated detectable doublet fraction = 66.6%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 35.9%
Elapsed time: 35.1 seconds
  -> Boundary for iteration 1: 0.1903
Iteration 2/10 - random_state: 176223
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.16
Detected doublet rate = 29.0%
Estimated detectable doublet fraction = 73.1%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 39.6%
Elapsed time: 35.7 seconds
  -> Boundary for iteration 2: 0.1862
Iteration 3/10 - random_state: 458165
Preprocessing...
Simulating doublets...
Embedding t

  -> Boundary for iteration 10: 0.2453

Final averaged doublet cut-off: 0.2441
Detected doublet rate = 24.3%
Estimated detectable doublet fraction = 62.3%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 39.0%
  ✅ CE357E1-S1- 处理完成！
     RNA 判定的 Doublet 数量: 3567
     ATAC 判定的 Doublet 数量: 3613
     两边共同判定的最终 Doublet: 2042
     结果已保存至: /mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/1_processed_data/CE357E1-S1-_Scrublet_Results.csv


正在处理样本: CE336E1-S1-
  原始 Barcode 总数: 625129
  初滤后(>=100 counts) 剩余真实细胞数: 3266
  --> 计算 RNA 模态的 Doublets...
Iteration 1/10 - random_state: 114716
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.17
Detected doublet rate = 22.9%
Estimated detectable doublet fraction = 78.7%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 29.1%
Elapsed time: 5.4 seconds
  -> Boundary for iteration 1: 0.2278
Iteration 2/10 - random_state: 112495
Preproce

Automatically set threshold at doublet score = 0.25
Detected doublet rate = 23.1%
Estimated detectable doublet fraction = 65.9%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 35.0%
Elapsed time: 27.3 seconds
  -> Boundary for iteration 9: 0.2393
Iteration 10/10 - random_state: 0
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.20
Detected doublet rate = 30.1%
Estimated detectable doublet fraction = 73.9%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 40.7%
Elapsed time: 28.5 seconds
  -> Boundary for iteration 10: 0.2393

Final averaged doublet cut-off: 0.2393
Detected doublet rate = 25.3%
Estimated detectable doublet fraction = 69.2%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 36.5%
  ✅ CE336E1-S1- 处理完成！
     RNA 判定的 Doublet 数量: 639
     ATAC 判定的 Doublet 数量: 825
     两边共同判定的最终 Doublet: 411
     结果已保存至: /mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_can

Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.42
Detected doublet rate = 5.1%
Estimated detectable doublet fraction = 34.9%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 14.5%
Elapsed time: 53.9 seconds
  -> Boundary for iteration 8: 0.2553
Iteration 9/10 - random_state: 644376
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.43
Detected doublet rate = 5.0%
Estimated detectable doublet fraction = 34.9%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 14.3%
Elapsed time: 56.7 seconds
  -> Boundary for iteration 9: 0.2553
Iteration 10/10 - random_state: 0
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.43
Detected doublet rate = 5.3%
Estimated detectable doublet fraction = 35.2%
Overall d

Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.06
Detected doublet rate = 56.0%
Estimated detectable doublet fraction = 94.4%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 59.3%
Elapsed time: 220.0 seconds
  -> Boundary for iteration 7: 0.1277
Iteration 8/10 - random_state: 479915
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.07
Detected doublet rate = 53.1%
Estimated detectable doublet fraction = 92.4%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 57.5%
Elapsed time: 227.1 seconds
  -> Boundary for iteration 8: 0.1296
Iteration 9/10 - random_state: 21868
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.06
Detected doublet rate = 55.8%
Estimat

...

  -> Boundary for iteration 5: 0.1728
Iteration 6/10 - random_state: 291262
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.14
Detected doublet rate = 41.6%
Estimated detectable doublet fraction = 82.3%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 50.6%
Elapsed time: 75.8 seconds
  -> Boundary for iteration 6: 0.1694
Iteration 7/10 - random_state: 985019
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.17
Detected doublet rate = 36.0%
Estimated detectable doublet fraction = 75.1%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 48.0%
Elapsed time: 78.3 seconds
  -> Boundary for iteration 7: 0.1728
Iteration 8/10 - random_state: 835894
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically 

  -> Boundary for iteration 4: 0.1827
Iteration 5/10 - random_state: 789587
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.27
Detected doublet rate = 22.2%
Estimated detectable doublet fraction = 54.4%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 40.8%
Elapsed time: 63.7 seconds
  -> Boundary for iteration 5: 0.1788
Iteration 6/10 - random_state: 930443
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.18
Detected doublet rate = 36.7%
Estimated detectable doublet fraction = 74.7%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 49.2%
Elapsed time: 82.9 seconds
  -> Boundary for iteration 6: 0.1827
Iteration 7/10 - random_state: 685142
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically 

  -> Boundary for iteration 3: 0.1836
Iteration 4/10 - random_state: 512669
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.07
Detected doublet rate = 52.2%
Estimated detectable doublet fraction = 92.4%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 56.4%
Elapsed time: 125.0 seconds
  -> Boundary for iteration 4: 0.1836
Iteration 5/10 - random_state: 64560
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.08
Detected doublet rate = 49.8%
Estimated detectable doublet fraction = 91.0%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 54.7%
Elapsed time: 142.4 seconds
  -> Boundary for iteration 5: 0.1836
Iteration 6/10 - random_state: 838090
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically

  -> Boundary for iteration 2: 0.2294
Iteration 3/10 - random_state: 314663
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.36
Detected doublet rate = 10.8%
Estimated detectable doublet fraction = 40.7%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 26.5%
Elapsed time: 74.6 seconds
  -> Boundary for iteration 3: 0.2234
Iteration 4/10 - random_state: 992799
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.36
Detected doublet rate = 10.7%
Estimated detectable doublet fraction = 40.8%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 26.3%
Elapsed time: 73.7 seconds
  -> Boundary for iteration 4: 0.2234
Iteration 5/10 - random_state: 695816
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically 

  -> Boundary for iteration 1: 0.2239
Iteration 2/10 - random_state: 378220
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.17
Detected doublet rate = 21.0%
Estimated detectable doublet fraction = 78.8%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 26.6%
Elapsed time: 64.7 seconds
  -> Boundary for iteration 2: 0.2239
Iteration 3/10 - random_state: 251069
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.17
Detected doublet rate = 20.2%
Estimated detectable doublet fraction = 78.1%
Overall doublet rate:
	Expected   = 15.0%
	Estimated  = 25.9%
Elapsed time: 63.0 seconds
  -> Boundary for iteration 3: 0.2308
Iteration 4/10 - random_state: 481164
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically 